In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [ ]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df.shape

(60000, 785)

In [ ]:
# train test split

X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    # Convert to PyTorch tensors
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
train_dataset = CustomDataset(X_train, y_train)

In [ ]:
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
len(train_loader)

1500

In [ ]:
class MyNN(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layers, dropout_rate):

    super().__init__()

    layers=[]

    for i in range(num_hidden_layers):
      layers.append(nn.Linear(input_dim, neurons_per_layers))
      layers.append(nn.BatchNorm1d(neurons_per_layers))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layers

    layers.append(nn.Linear(neurons_per_layers, output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)

In [ ]:
# objective function
def objective(trial):

  # next hyperparameter value from the search space
  num_hidden_layers = trial.suggest_int('num_hidden_layers',1,5)
  neurons_per_layer = trial.suggest_int('neurons_per_layer',8,128, step=8)
  epochs = trial.suggest_int('epochs', 10, 50, step=10)
  learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
  dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical('batch_size',[16,32,64,128])
  optimizer_name = trial.suggest_categorical('optimizer', ['Adam','SGD','RMSprop'])
  weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)


  train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

  # model initialize
  input_dim = 784
  output_dim = 10

  model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model.to(device)

  # loss function
  criterion = nn.CrossEntropyLoss()

  # optimizer
  if optimizer_name == 'Adam':
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # training loop
  for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:

      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # calculate loss
      loss = criterion(outputs, batch_labels)

      # backward pass
      optimizer.zero_grad()
      loss.backward()

      # update gradients
      optimizer.step()

  # evaluation
  model.eval()

  # evaluation on test data
  total=0
  correct=0

  with torch.no_grad():
    for batch_features, batch_labels in test_loader:
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs.data, 1)

      total += batch_labels.shape[0]
      correct += (predicted == batch_labels).sum().item()

    accuracy = correct/total

  return accuracy





In [ ]:
# Installing Optuna
!pip install optuna

In [ ]:
import optuna

In [ ]:
# Create study
study = optuna.create_study(direction='maximize')

[I 2026-03-02 18:40:39,564] A new study created in memory with name: no-name-8c286114-0a93-46c6-a640-54911af9f8bc


In [ ]:
# running the study
study.optimize(objective, n_trials=10) # n_trials=10 means basically we are training the model for about 10 times to get the best hyperparameter values.

[I 2026-03-02 18:46:26,225] Trial 0 finished with value: 0.1035 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 96, 'epochs': 50, 'learning_rate': 0.08730402248370878, 'dropout_rate': 0.4, 'batch_size': 64, 'optimizer': 'RMSprop', 'weight_decay': 0.00023301329957099313}. Best is trial 0 with value: 0.1035.
[I 2026-03-02 18:48:11,595] Trial 1 finished with value: 0.7303333333333333 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 120, 'epochs': 30, 'learning_rate': 0.028551314072717774, 'dropout_rate': 0.4, 'batch_size': 128, 'optimizer': 'RMSprop', 'weight_decay': 0.00043722449302864065}. Best is trial 1 with value: 0.7303333333333333.
[I 2026-03-02 18:48:44,783] Trial 2 finished with value: 0.6855833333333333 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 88, 'epochs': 20, 'learning_rate': 1.1595873026902327e-05, 'dropout_rate': 0.1, 'batch_size': 32, 'optimizer': 'SGD', 'weight_decay': 0.0005647414231765227}. Best is trial 1 with value: 0.730

In [ ]:
# best accuracy
study.best_value

0.8394166666666667

In [ ]:
# best hyperparameters
study.best_params

{'num_hidden_layers': 5,
 'neurons_per_layer': 104,
 'epochs': 30,
 'learning_rate': 0.006071820562257154,
 'dropout_rate': 0.4,
 'batch_size': 128,
 'optimizer': 'Adam',
 'weight_decay': 7.930056723919331e-05}

In [ ]:
# The architecture would work like= (784,96)>(96,96)>(96,96)>(96,96)>(96,10)